# Análise de dados

In [ ]:
# Importa as bibliotecas para análise de dados e requisições à API
import pandas as pd
import requests

# Define o endereço da API
url_base = "http://127.0.0.1:5000"

# Cria uma sessão para manter o login e os cookies de autenticação
sessao = requests.Session()

# Realiza o login de administrador na API
resposta_login = sessao.post(
    f"{url_base}/login",
    json={
        "usuario": "admin",
        "senha": "admin123"
    }
)

# Exibe o status do login e verifica se ocorreu algum erro
print("Login:", resposta_login.status_code)
print(resposta_login.text)
resposta_login.raise_for_status()

# Busca os registros de acesso utilizando a sessão autenticada
resposta = sessao.get(f"{url_base}/acessos")

# Exibe o status da consulta e verifica se a requisição foi bem-sucedida
print("Acessos:", resposta.status_code)
resposta.raise_for_status()

# Converte os dados retornados pela API em um DataFrame para análise
dados = pd.DataFrame(resposta.json())

# Mostra as primeiras linhas dos registros consultados
dados.head()

Login: 200
{"mensagem":"Login realizado com sucesso."}

Acessos: 200


,evento,id,matricula,nome,registrado_em,resultado,uid
0,entrada,1,None,None,2026-10-28 12:09:04,permitido,A1B2C3D4


In [ ]:
# Converte a coluna de registro para o formato de data e hora e cria uma nova coluna com apenas a data

dados["registrado_em"] = pd.to_datetime(dados["registrado_em"])

dados["data"] = dados["registrado_em"].dt.strftime("%Y-%m-%d")

In [ ]:
# Selecionando os registros do dia que será analisado

dia = "2026-10-28"

dados_dia = dados[dados["data"] == dia]

print("Data analisada:")
print(dia)

print("Total de registros no dia:")
print(dados_dia.shape[0])


Data analisada:
2026-10-28
Total de registros no dia:
1


In [ ]:
# Quantas pessoas entraram na sala em determinado dia?

print("Registros do dia:")
display(dados_dia[["nome", "evento", "resultado", "registrado_em"]])

entradas = dados_dia[
    (dados_dia["evento"] == "entrada") &
    (dados_dia["resultado"] == "permitido")
]

print("Total de entradas permitidas:", len(entradas))

# Contando quantos colaboradores diferentes entraram na sala no dia

Todos os registros:


,nome,evento,resultado,registrado_em
0,None,entrada,permitido,2026-10-28 12:09:04


Registros do dia:


,nome,evento,resultado,registrado_em
0,None,entrada,permitido,2026-10-28 12:09:04


Total de entradas permitidas: 1


In [14]:
# Quantas pessoas saíram da sala em determinado dia?

saidas = dados_dia[
    (dados_dia["evento"] == "saida") &
    (dados_dia["resultado"] == "permitido")
]

print("Total de saídas permitidas:", len(saidas))
display(saidas)

# Contando quantos colaboradores diferentes saíram da sala no dia

Total de saídas permitidas: 0


,evento,id,matricula,nome,registrado_em,resultado,uid,data


In [15]:
# Quanto tempo um colaborador específico permaneceu na sala em um determinado dia?

matricula = "MAT001"

colaborador = dados_dia[dados_dia["matricula"] == matricula]

entradas = colaborador[
    colaborador["evento"] == "entrada"
]["registrado_em"].tolist()

saidas = colaborador[
    colaborador["evento"] == "saida"
]["registrado_em"].tolist()

tempo_total = pd.Timedelta(0)

for entrada, saida in zip(entradas, saidas):
    if saida >= entrada:
        tempo_total += saida - entrada

print("Matrícula:", matricula)
print("Dia:", dia)
print("Tempo total na sala:", tempo_total)

# Calculando o tempo de permanência do colaborador no dia

Matrícula: MAT001
Dia: 2026-10-28
Tempo total na sala: 0 days 00:00:00


In [16]:
# Quantas tentativas de acesso negado ocorreram em um determinado dia?

print("Tentativas de acesso negado:")
print(
    dados_dia[dados_dia["resultado"] == "negado"].shape[0]
)

# Contando as tentativas de acesso negado no dia

Tentativas de acesso negado:
0


In [17]:
# Quantas tentativas de invasão ocorreram em um determinado dia?

print("Tentativas com tags desconhecidas:")
print(
    dados_dia[dados_dia["evento"] == "desconhecido"].shape[0]
)

# Contando os eventos de tags desconhecidas no dia

Tentativas com tags desconhecidas:
0


In [21]:
# Quais colaboradores não autorizados mais tentaram acessar a sala em um determinado dia?

nao_autorizados = dados_dia[
    dados_dia["resultado"] == "negado"
]

if nao_autorizados.empty:
    print("Não houve tentativas de acesso negadas neste dia.")
else:
    print("Tentativas de acesso negadas:")
    display(nao_autorizados)

# Identificando quais colaboradores não autorizados mais tentaram acessar a sala

Não houve tentativas de acesso negadas neste dia.
